In [2]:
import os
import json
import time
from typing import cast

from dotenv import load_dotenv
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import (
    AnalysisInput,
    AnalysisResult,
    AudioVisualContent,
    DocumentContent,
    AnalysisContent,
)
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from IPython.display import display, Markdown, Latex

load_dotenv()

endpoint = os.environ["AZURE_AI_ENDPOINT"]
key = os.getenv("AZURE_AI_API_KEY")
credential = AzureKeyCredential(key) if key else DefaultAzureCredential()

client = ContentUnderstandingClient(endpoint=endpoint, credential=credential)

In [ ]:
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer,
    ContentAnalyzerConfig,
    ContentFieldSchema,
    ContentFieldDefinition,
    ContentFieldType,
    GenerationMethod,
)

# Generate a unique analyzer ID
analyzer_id = f"my_document_analyzer_{int(time.time())}"

# Define field schema with custom fields
field_schema = ContentFieldSchema(
    name="InvoiceFields",
    description="Schema for extracting Invoice information",
    fields={
        "VendorName": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.EXTRACT,
            description="Name of the vendor or supplier, typically found in the header section",
            estimate_source_and_confidence=True,            
        ),
        "InvoiceItems": ContentFieldDefinition(
            type=ContentFieldType.ARRAY,
            method=GenerationMethod.GENERATE,
            description="List of items or services billed in the invoice, typically found in the line items section",
            estimate_source_and_confidence=True,
            item_definition=ContentFieldDefinition(
                type=ContentFieldType.OBJECT,
                properties= {
                    "description": ContentFieldDefinition(
                        type=ContentFieldType.STRING,   
                        description="Description of the billed item or service",
                    ),
                    "quantity": ContentFieldDefinition(
                        type=ContentFieldType.NUMBER,   
                        description="Quantity of the billed item or service",
                    ),
                }
            )
        )
    }
)

# Create analyzer configuration
config = ContentAnalyzerConfig(
    enable_formula=True,
    enable_layout=True,
    enable_ocr=True,
    estimate_field_source_and_confidence=True,
    return_details=True,
)

# Create the analyzer with field schema
analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",
    description=(
        "Custom analyzer for extracting company information"
    ),
    config=config,
    field_schema=field_schema,
    models={
        "completion": "gpt-4.1",
        "embedding": "text-embedding-3-large",
    }, # Required when using field_schema and prebuilt-document base analyzer
)

# Create the analyzer
poller = client.begin_create_analyzer(
    analyzer_id=analyzer_id,
    resource=analyzer,
)
result = poller.result() # Wait for creation to complete

# Get the full analyzer details after creation
result = client.get_analyzer(analyzer_id=analyzer_id)
print(f"Analyzer '{analyzer_id}' created successfully!")

if result.description:
    print(f"  Description: {result.description}")

if result.field_schema and result.field_schema.fields:
    print(f"  Fields ({len(result.field_schema.fields)}):")
    for field_name, field_def in result.field_schema.fields.items():
        method = field_def.method if field_def.method else "auto"
        field_type = field_def.type if field_def.type else "unknown"
        print(f"    - {field_name}: {field_type} ({method})")

HttpResponseError: (InvalidRequest) Invalid Request.
Code: InvalidRequest
Message: Invalid Request.
Inner error: {
    "code": "InvalidFieldSchema",
    "message": "One or more errors encountered while processing the field schema.",
    "details": [
        {
            "code": "MissingProperty",
            "message": "The 'items' property is required but is currently missing.",
            "target": "/fieldSchema/fields/InvoiceItems"
        }
    ]
}